# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 Croissant dataset using the `mlcroissant` library. All Croissant entities (record sets, fields, columns) are referenced by their `@id` fields for reproducibility and clarity.

### Dataset Source
The dataset is described and structured via a Croissant schema accessible at the following URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the FAIR^2 Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Authors: {getattr(metadata, 'author', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")
print(f"Cite as: {getattr(metadata, 'citeAs', None)}")

## 2. Data Overview
Explore available **record sets** (tables), their **fields** (columns), and all relevant `@id` values as defined by the Croissant schema. This will help us understand the dataset structure.


In [ ]:
# List all available record sets and their fields by @id
record_sets = []
print("Available record sets (by @id and name):\n")
for recset in getattr(metadata, 'recordSet', []):
    print(f"- @id: {recset['@id']}   name: {recset.get('name', '[no name]')}")
    record_sets.append(recset['@id'])

# For demonstration, print fields in the first record set (if any)
if record_sets:
    first_recset = None
    for recset in getattr(metadata, 'recordSet', []):
        if recset['@id'] == record_sets[0]:
            first_recset = recset
            break
    if first_recset:
        print(f"\nFields (by @id) in record set {record_sets[0]}:")
        for field in first_recset.get('field', []):
            print(f"- {field['@id']}: {field.get('name', '[no name]')} (data type: {field.get('dataType', '[n/a]')})")


In [ ]:
# Preview records in the first available record set by @id
if record_sets:
    example_record_set_id = record_sets[0]
    print(f"\nPreviewing first 2 records from record set: {example_record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        pprint(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for further analysis. You can reference the desired record set(s) by their `@id` as shown above.


In [ ]:
# Extract all record sets into a dictionary of DataFrames, using @id as the key
dataframes = {}

# For demonstration, extract only the first record set (feel free to expand to all)
for recset in getattr(metadata, 'recordSet', []):
    recset_id = recset['@id']
    print(f"Loading records from record set: {recset_id} ...")
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Shape: {df.shape}\n")
    else:
        print("  No records found.\n")

# Pick the first, or explicitly set the main record set id for processing below
if dataframes:
    main_record_set_id = list(dataframes)[0]
    print(f"Example: showing head of record set {main_record_set_id}")
    display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply basic transformations such as filtering numeric values, normalization, and grouping. **All field/column operations refer to columns by their original Croissant field `@id`**.

In [ ]:
import numpy as np

# Use the first loaded DataFrame for EDA
df = dataframes[main_record_set_id]
print(f"Using DataFrame for record set: {main_record_set_id}\nColumns:\n{df.columns.tolist()}")

# Identify a likely numeric field by checking dtypes and column names
print("\nInferring likely numeric fields:")
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
print(numeric_columns)

# If numeric columns exist, pick the first for demo
if numeric_columns:
    numeric_field = numeric_columns[0]
    print(f"\nAnalyzing numeric field: {numeric_field}")
    
    # Filter: show records where numeric_field > mean
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - df[numeric_field].mean()) / df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Attempt group by a likely category field
    non_numeric_columns = [col for col in df.columns if col not in numeric_columns and df[col].dtype == object]
    print(f"\nNon-numeric columns: {non_numeric_columns}")
    if non_numeric_columns:
        group_field = non_numeric_columns[0]
        print(f"Grouping by field: {group_field}")
        grouped = filtered_df.groupby(group_field)[numeric_field].mean()
        print("Grouped mean values after filtering and normalization:")
        print(grouped.head())
else:
    print('No numeric fields available for EDA.')

## 5. Visualization
Visualize data distributions and relationships using pandas and matplotlib. Below, we'll create a histogram for a numeric field, boxplot, and, if possible, a category mean plot.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_columns and len(df) > 0:
    # Histogram
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # Boxplot (for all records)
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[numeric_field])
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()
    
    # If we have a suitable group_field from before
    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        sns.barplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"{numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- Successfully loaded and identified record sets and fields using Croissant `@id` semantics.
- Extracted tabular records into pandas DataFrames for programmatic analysis.
- Performed filtering, normalization, and grouped aggregations using numeric and categorical fields by their canonical Croissant `@id`-based column names.
- Visualized field distributions and relationships. Further domain analysis can proceed using the references and structure provided in the FAIR^2 schema.